# 06 - Lattice matching

TrackPad exposes tracking and optics primitives rather than prescribing an optimizer. Here a two-family Newton match uses a finite-difference tune response.

In [ ]:
import Pkg
EXAMPLES_DIR = isfile(joinpath(pwd(), "common.jl")) ? pwd() : joinpath(pwd(), "examples")
Pkg.activate(EXAMPLES_DIR)
using TrackPad, StaticArrays
include(joinpath(EXAMPLES_DIR, "common.jl"))
using .TrackPadExamples

using LinearAlgebra

In [ ]:
base_ring, beam = madx_fodo()
target = collect(gettune(base_ring, beam))
strengths = [1.2, -1.2]

model(k) = set_fodo_strengths(base_ring; qf=k[1], qd=k[2])
observables(k) = collect(gettune(model(k), beam))
for iteration in 1:6
    residual = observables(strengths) - target
    norm(residual) < 1e-11 && break
    h = 1e-5
    response = hcat(((observables(strengths + h .* [j == i for j in 1:2]) -
                       observables(strengths - h .* [j == i for j in 1:2])) / (2h)
                     for i in 1:2)...)
    strengths .-= response \ residual
end

(target=target, matched=gettune(model(strengths), beam),
 strengths=strengths)

For production matching, replace the Newton step with the optimizer and bounds appropriate to the machine. Keep the objective as a function that constructs immutable elements and returns TrackPad observables.